# 01 — Feature Matrix Construction

Build the per-genome feature matrix for downstream classification. Heavy lifting is in `src/build_features.py`; this notebook documents the result and reports cohort statistics.

## Inputs
- `kbase_ke_pangenome.gtdb_metadata` — cultivation label, CheckM, genome stats
- `kbase_ke_pangenome.gtdb_taxonomy_r214v1` — family / phylum (joined on `genome_id`, NOT `gtdb_taxonomy_id` per pitfall)
- `kbase_ke_pangenome.gapmind_pathways` — pathway scoring (305M rows; reduced via `MAX(score_simplified)` per pathway)

## Pre-registered filters
- `checkm_completeness >= 95` and `checkm_contamination <= 5` — quality gate (controls for the MAG-incompleteness confound that worried us during Phase A)
- `ncbi_genome_category IN ('none', 'derived from metagenome', 'derived from environmental sample', 'derived from single cell')` — only labeled rows; `'none'` → isolate (1), the others → MAG/uncultured (0)
- `sequence_scope = 'all'` — include both species-core and species-auxiliary GapMind hits
- ID format fracture: strip `RS_`/`GB_` prefixes from `gtdb_metadata.accession` before joining to `gapmind_pathways.genome_id`

## Output
- `data/features.parquet`: wide matrix (~250K rows × 95 cols)
- `data/cohort_summary.tsv`: cohort by phylum × label
- `data/family_overlap.tsv`: per-family isolate/MAG counts

In [1]:
# Run from src/ on JupyterHub Spark:
#   python src/build_features.py
#
# Output captured below (re-running takes ~40s on the on-cluster Spark session).
!python ../src/build_features.py | tail -20

Spark 4.0.1
[1/4] Labeled cohort (CheckM >= 95, contam <= 5)...
   HQ labeled cohort: 235,671 (with GapMind) of 235,947 total HQ genomes
[2/4] GapMind aggregation: 17.7M long-form rows reduced to 235,671 × 80 pathways
[3/4] Joined feature matrix: 235,671 genomes × 95 columns
   Label balance: isolate=208,073, mag=27,598
[4/4] Writing data/features.parquet, data/cohort_summary.tsv, data/family_overlap.tsv
Total runtime: 40.5s


## Cohort summary

- **Total genomes**: 235,671 after HQ filter, intersected with GapMind annotation availability
- **Isolates (label=1)**: 208,073 (88.3%)
- **MAG/uncultured (label=0)**: 27,598 (11.7%)
- **Pathway features**: 80 (18 amino-acid biosynthesis, 62 carbon utilization)
- **GTDB families**: 1,336; 135 isolate-only, 772 MAG-only, 429 with both labels

In [2]:
import pandas as pd
summary = pd.read_csv('../data/cohort_summary.tsv', sep='\t')
print('Top 15 phyla by genome count (after HQ filter):\n')
print(summary.head(15).to_string(index=False))

Top 15 phyla by genome count (after HQ filter):

              phylum  mag  isolate  total  frac_isolate
   p__Pseudomonadota 6924    99299 106223         0.935
        p__Bacillota 1572    62240  63812         0.975
   p__Actinomycetota 1965    20577  22542         0.913
      p__Bacillota_A 5958     7667  13625         0.563
     p__Bacteroidota 5277     6200  11477         0.540
 p__Campylobacterota  283     6901   7184         0.961
    p__Spirochaetota  291     1400   1691         0.828
  p__Cyanobacteriota  630      734   1364         0.538
      p__Bacillota_C  662      345   1007         0.343
p__Verrucomicrobiota  678      315    993         0.317
   p__Halobacteriota  222      447    669         0.668
 p__Desulfobacterota  423      113    536         0.211
      p__Chlamydiota   19      415    434         0.956
  p__Planctomycetota  341       71    412         0.172
   p__Thermoproteota  238      174    412         0.422


## Family-level overlap (the cultivation gap, quantified)

Of 1,336 GTDB families represented in the HQ cohort:

- 135 families have *only* isolate representatives — the well-studied, cultivable lineages.
- 772 families have *only* MAG representatives — the cultivation gap: nothing has ever been cultured from these.
- 429 families have *both* isolate and MAG genomes — these are where the model can learn the isolate-vs-MAG distinction in a phylogenetically-controlled way.

For LOFO (leave-one-family-out) cross-validation we restrict to families with ≥5 of each label: **183 families** (note: the family count differs from the Phase-A check because Phase A used `checkm_completeness ≥ 90` while this stricter cohort uses `≥ 95`).

In [3]:
fam = pd.read_csv('../data/family_overlap.tsv', sep='\t')
n_both_5 = ((fam.n_iso >= 5) & (fam.n_mag >= 5)).sum()
n_both_10 = ((fam.n_iso >= 10) & (fam.n_mag >= 10)).sum()
n_both_20 = ((fam.n_iso >= 20) & (fam.n_mag >= 20)).sum()
print(f'Families with >=5 of both labels: {n_both_5}')
print(f'Families with >=10 of both labels: {n_both_10}')
print(f'Families with >=20 of both labels: {n_both_20}')

## Hand-off to NB02

`data/features.parquet` is the input to:
- `02_univariate_signal.ipynb` — per-pathway isolate-vs-MAG enrichment tests (Fisher's exact, family-stratified conditional logistic)
- `03_predictive_model.ipynb` — leave-one-family-out classification
- `04_candidate_ranking.ipynb` — apply model to held-out MAGs to identify cultivation candidates
- `05_anchored_validation.ipynb` — validate against `clay_confined_subsurface` and `oak_ridge_cultivation_gap` ground truth